# ClimaCity Paris -- 00. Configuration

Ce carnet prépare et vérifie l'environnement avant les six carnets du projet :

1. il charge la **configuration partagée** (`scripts/config.py`) : chemins, paramètres Spark, graine ;
2. il **collecte les données** si elles manquent (`scripts/collecte.py`, idempotent) ;
3. il vérifie les **versions** des outils et l'accès au serveur **MLflow** ;
4. il ouvre une **SparkSession** et teste **Delta Lake** (écriture, time travel, `MERGE`).

À exécuter une fois après `docker compose up -d --build`, puis chaque fois que l'environnement change.

## 1. Configuration partagée

Tous les carnets commencent par la même cellule : ils ajoutent `../scripts` au chemin de Python puis
importent `config`. Les chemins sont donc déclarés **une seule fois**, relativement à la racine du projet
(contrainte n°2 : aucun chemin absolu en dur).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..", "scripts").resolve()))
from config import *          # chemins, SEED, SHUFFLE_PARTITIONS, creer_session()
import config

for nom in ["RACINE", "VELIB_RAW", "GBFS_JSON", "METEO_CSV", "AIR_RAW", "SORTIES"]:
    print(f"{nom:<20} {getattr(config, nom)}")
print(f"{'SEED':<20} {SEED}")
print(f"{'SHUFFLE_PARTITIONS':<20} {SHUFFLE_PARTITIONS}")
print(f"{'DRIVER_MEMORY':<20} {DRIVER_MEMORY}   (vient du .env)")
print(f"{'MLFLOW_URI':<20} {MLFLOW_URI}")

RACINE               /home/jovyan
VELIB_RAW            /home/jovyan/data/velib/raw
GBFS_JSON            /home/jovyan/data/velib/station_information.json
METEO_CSV            /home/jovyan/data/meteo/meteo_montsouris_horaire.csv
AIR_RAW              /home/jovyan/data/air/raw
SORTIES              /home/jovyan/data/output
SEED                 42
SHUFFLE_PARTITIONS   8
DRIVER_MEMORY        4g   (vient du .env)
MLFLOW_URI           http://mlflow:5000


## 2. Collecte des données

`collecter()` ne télécharge que ce qui manque : la première exécution prend environ 4 minutes
(~500 Mo), les suivantes une seconde. Le script ne fait que de l'**acquisition** ; tout le traitement se
fait ensuite en Spark.

In [2]:
import time
import collecte

t0 = time.perf_counter()
resume = collecte.collecter(DATA)
print(f"Collecte en {time.perf_counter() - t0:.1f} s")
for cle, valeur in resume.items():
    print(f"  {cle:<15} {valeur}")

# Contrôle : chaque source attendue est présente et non vide.
attendus = {
    "Vélib' (csv.gz)": list(VELIB_RAW.glob("velib_*.csv.gz")),
    "GBFS": [GBFS_JSON],
    "Météo": [METEO_CSV],
    "Qualité de l'air (parquet)": list(AIR_RAW.glob("*.parquet")),
}
for source, fichiers in attendus.items():
    taille = sum(f.stat().st_size for f in fichiers) / 1_048_576
    assert fichiers and all(f.exists() for f in fichiers), f"Source manquante : {source}"
    print(f"[OK] {source:<28} {len(fichiers):>3} fichier(s)  {taille:>7.1f} Mo")

Collecte en 0.3 s
  periode         ('2020-11-26', '2021-04-09')
  fichiers_velib  10
  gbfs            /home/jovyan/data/velib/station_information.json
  meteo           /home/jovyan/data/meteo/meteo_montsouris_horaire.csv
  fichiers_air    50
[OK] Vélib' (csv.gz)               10 fichier(s)    221.2 Mo
[OK] GBFS                           1 fichier(s)      0.3 Mo
[OK] Météo                          1 fichier(s)      0.1 Mo
[OK] Qualité de l'air (parquet)    50 fichier(s)     61.4 Mo


## 3. Versions et services

L'énoncé demande PySpark 3.5 et une version de `delta-spark` compatible (Delta 3.2 pour Spark 3.5).

In [3]:
import platform
import subprocess
import importlib.metadata as md
import requests

java = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0]
versions = {
    "Python": platform.python_version(),
    "Java": java,
    "pyspark": md.version("pyspark"),
    "delta-spark": md.version("delta-spark"),
    "mlflow": md.version("mlflow"),
    "folium": md.version("folium"),
    "plotly": md.version("plotly"),
}
for outil, v in versions.items():
    print(f"{outil:<12} {v}")
assert versions["pyspark"].startswith("3.5"), "PySpark 3.5 attendu"
assert versions["delta-spark"].startswith("3.2"), "delta-spark 3.2 attendu avec Spark 3.5"

# Le serveur MLflow tourne dans son propre conteneur ; Jupyter le joint par le réseau Docker.
try:
    etat = requests.get(f"{MLFLOW_URI}/health", timeout=5).text
except requests.RequestException as e:
    etat = f"injoignable ({type(e).__name__}) : les runs iront dans {SORTIES / 'mlruns'}"
print(f"\nServeur MLflow {MLFLOW_URI} : {etat}")

Python       3.11.10
Java         openjdk version "17.0.12" 2024-07-16
pyspark      3.5.3
delta-spark  3.2.0
mlflow       2.17.2
folium       0.17.0
plotly       5.24.1

Serveur MLflow http://mlflow:5000 : OK


## 4. SparkSession et test de Delta Lake

`creer_session()` fixe le mode `local[*]` (contrainte n°1), 8 partitions de shuffle au lieu de 200, le
fuseau UTC et les extensions Delta. Le test ci-dessous écrit une petite table Delta, la modifie par
`MERGE`, puis relit sa première version : les trois opérations du Jour 2 fonctionnent.

In [4]:
spark = creer_session("ClimaCity-00-configuration")
sc = spark.sparkContext

print(f"Spark {spark.version} | master {sc.master} | {sc.defaultParallelism} coeurs")
print(f"shuffle.partitions = {spark.conf.get('spark.sql.shuffle.partitions')} | "
      f"fuseau = {spark.conf.get('spark.sql.session.timeZone')} | "
      f"mémoire driver = {spark.conf.get('spark.driver.memory')}")
print(f"Spark UI : {SPARK_UI}")

Spark 3.5.3 | master local[*] | 8 coeurs


shuffle.partitions = 8 | fuseau = UTC | mémoire driver = 4g
Spark UI : http://localhost:4040


In [5]:
from delta.tables import DeltaTable

chemin_test = str(SORTIES / "test_delta")
spark.range(5).withColumnRenamed("id", "cle").write.format("delta").mode("overwrite").save(chemin_test)

(
    DeltaTable.forPath(spark, chemin_test).alias("cible")
    .merge(spark.range(3, 8).withColumnRenamed("id", "cle").alias("source"), "cible.cle = source.cle")
    .whenNotMatchedInsertAll()
    .execute()
)
n_v0 = spark.read.format("delta").option("versionAsOf", 0).load(chemin_test).count()
n_actuel = spark.read.format("delta").load(chemin_test).count()
print(f"Version 0 : {n_v0} lignes | après MERGE : {n_actuel} lignes")
assert (n_v0, n_actuel) == (5, 8), "Delta Lake ne se comporte pas comme prévu"

import shutil
shutil.rmtree(chemin_test)
print("Delta Lake : écriture, MERGE et time travel OK")

Version 0 : 5 lignes | après MERGE : 8 lignes
Delta Lake : écriture, MERGE et time travel OK


## 5. Bilan

Si toutes les cellules passent, l'environnement est prêt : les carnets `01` à `06` peuvent s'exécuter
dans l'ordre. On libère la session pour rendre la mémoire au conteneur.

In [6]:
spark.stop()
print("SparkSession arrêtée.")

SparkSession arrêtée.
